# PEGASUS on SAMSum ? small Colab efficiency update
Select **Runtime > Change runtime type > GPU**, then run the cells in order.
The model, full training/validation datasets, three epochs, learning rate, and token limits are preserved.

Changes: batch size 2 with accumulation 2 (effective batch remains 4), batches grouped by input length, padding to multiples of 8, BF16 where supported (FP16 otherwise), and preprocessing only the splits used in training. Adafactor and gradient checkpointing remain enabled.

If batch size 2 exceeds GPU memory, restart the session and change `BATCH_SIZE` to `1`. Actual speed depends on the allocated GPU; no speedup has been measured here.

Run the final save cell after training. Set `DOWNLOAD_MODEL = True` to download the export. Files under `/content` are temporary: download the model or copy it to Drive before ending the session. Extract the final model files into your local `models/pegasus_samsum` folder to use with your project's existing configuration.

The setup cell uses Transformers 4.x because this notebook uses its `group_by_length` training option.
References: [Trainer](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer), [data collator](https://huggingface.co/docs/transformers/main_classes/data_collator).


In [ ]:
%pip install -q "transformers>=4.46,<5" datasets accelerate sentencepiece


In [ ]:
import torch

print("Torch:", torch.__version__)
assert torch.cuda.is_available(), "In Colab select Runtime > Change runtime type > GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
USE_BF16 = torch.cuda.is_bf16_supported()
print("Precision:", "BF16" if USE_BF16 else "FP16")


In [ ]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

In [ ]:
MODEL_NAME = "google/pegasus-xsum"
DATASET_NAME = "knkarthick/samsum"
EPOCHS = 3

# Two examples per forward pass; effective batch size stays at four.
# If GPU memory runs out, restart the session and set BATCH_SIZE = 1.
BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 4 // BATCH_SIZE
LEARNING_RATE = 0.00005
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128
MODEL_DIR = "/content/pegasus_samsum"


In [ ]:
dataset = load_dataset(DATASET_NAME)

print(dataset)

In [ ]:
print("Dialogue:")
print(dataset["train"][0]["dialogue"])

print("\nSummary:")
print(dataset["train"][0]["summary"])

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

print("Tokenizer loaded successfully")

In [ ]:
def preprocess_function(batch):

    # Dialogue -> Encoder input
    model_inputs = tokenizer(
        batch["dialogue"],
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    # Summary -> Decoder target
    labels = tokenizer(
        text_target=batch["summary"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
from datasets import DatasetDict

# The test split is not used during training; leave it untouched for later evaluation.
tokenized_dataset = DatasetDict({
    split: dataset[split].map(
        preprocess_function,
        batched=True,
        remove_columns=dataset[split].column_names,
        desc=f"Tokenizing {split}",
    )
    for split in ("train", "validation")
})
print(tokenized_dataset)


In [ ]:
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

# Important when using gradient checkpointing
model.config.use_cache = False

print("Pegasus model loaded successfully")

In [ ]:
total_parameters = sum(
    p.numel() for p in model.parameters()
)

print(
    "Parameters:",
    f"{total_parameters:,}"
)

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,  # Align padded batches for GPU Tensor Cores.
)


In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=MODEL_DIR,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    bf16=USE_BF16,
    fp16=not USE_BF16,
    optim="adafactor",
    group_by_length=True,  # Batch similar-length inputs to reduce padding.
    prediction_loss_only=True,  # Validation needs loss, not stored vocabulary logits.
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    report_to="none",
    predict_with_generate=False,
    seed=42,
)


In [ ]:
trainer = Seq2SeqTrainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset["train"],

    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator,

    processing_class=tokenizer
)

In [ ]:
trainer.train()

In [ ]:
# Save a standalone model and tokenizer for your project.
# Checkpoints remain in MODEL_DIR; the final export has its own folder.
from pathlib import Path

FINAL_MODEL_DIR = str(Path(MODEL_DIR) / "final_model")
model.config.use_cache = True
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("Saved model and tokenizer to:", FINAL_MODEL_DIR)

# Set True to download the final model after training.
DOWNLOAD_MODEL = False
if DOWNLOAD_MODEL:
    import shutil
    from google.colab import files
    archive = shutil.make_archive("/content/pegasus_samsum_final", "zip", FINAL_MODEL_DIR)
    files.download(archive)
